### 01. Get Federal Register rules about AI

**Goal:** download every final rule published since January 20, 2017 that uses an AI-related search term, label each rule with the administration that published it, and save the results as a table.

**Saves:** `fr_rules.csv` (one row per rule)

**Next notebook:** `02_llmjudge.ipynb`

**AI assistance:** The code in this notebook was drafted alongside Claude and was reviewed, run, and checked by our team.

Links to official documentation are in the last cell.

#### Step 1: Import libraries

In [1]:
import requests
import pandas as pd

#### Step 2: Settings

The Federal Register API needs no account or key. All the values that might change are kept in this one cell.


In [ ]:
# Web address of the Federal Register search API
api_url = "https://www.federalregister.gov/api/v1/documents.json"

# Start date: Trump's first inauguration
start_date = "2017-01-20"

# Search terms
related_terms = [
    "artificial intelligence",
    "machine learning",
    "automated decision",
    "deep learning",
    "large language model",
]

# The information to return for each rule
fields = ["document_number", "title", "abstract", "publication_date",
          "agencies", "html_url", "raw_text_url"]

#### Step 3: Download the rules for every search term

Step 3a searches for one term to show what a single request looks like. Step 3b repeats the same search for every term.

##### Step 3a: Search for one term "artificial intelligence"

In [11]:
# Search settings, stored as a dictionary
params = {
    "conditions[term]": '"artificial intelligence"',      # quotes = the words must appear next to each other
    "conditions[type][]": "RULE",                         # final rules only
    "conditions[publication_date][gte]": start_date,      # gte = on or after this date
    "per_page": 1000,                        # return up to 1,000 rules
    "fields[]": fields,                      # information to return for each rule
}

response = requests.get(api_url, params=params, timeout=60)   # send the request to the Federal Register and wait up to 60 seconds for a reply
response.raise_for_status()          # stop with an error if the request failed
data = response.json()               # turn the reply into a Python dictionary

print("Rules found:", data["count"]) # how many rules match the search
pd.DataFrame(data["results"]).head() # show the first 5 rules as a table

Rules found: 190


,document_number,title,abstract,publication_date,agencies,html_url,raw_text_url
0,2025-00636,Framework for Artificial Intelligence Diffusion,"With this interim final rule, the Commerce Dep...",2025-01-15,"[{'raw_name': 'DEPARTMENT OF COMMERCE', 'name'...",https://www.federalregister.gov/documents/2025...,https://www.federalregister.gov/documents/full...
1,2025-00637,Public Briefing on Framework for Artificial In...,"On January 10, 2025, the Office of the Federal...",2025-01-15,"[{'raw_name': 'DEPARTMENT OF COMMERCE', 'name'...",https://www.federalregister.gov/documents/2025...,https://www.federalregister.gov/documents/full...
2,2023-05321,Copyright Registration Guidance: Works Contain...,The Copyright Office issues this statement of ...,2023-03-16,"[{'raw_name': 'LIBRARY OF CONGRESS', 'name': '...",https://www.federalregister.gov/documents/2023...,https://www.federalregister.gov/documents/full...
3,2026-07087,Final Priority and Definitions-Secretary's Sup...,The Department of Education (Department) annou...,2026-04-13,"[{'raw_name': 'DEPARTMENT OF EDUCATION', 'name...",https://www.federalregister.gov/documents/2026...,https://www.federalregister.gov/documents/full...
4,2026-22593,Medical Devices; Hematology and Pathology Devi...,The Food and Drug Administration (FDA) is clas...,2026-11-05,[{'raw_name': 'DEPARTMENT OF HEALTH AND HUMAN ...,https://www.federalregister.gov/documents/2026...,https://www.federalregister.gov/documents/full...


##### Step 3b: Search for every term

The same search runs for each term. Each term's rules become a small table, and the tables are stacked into one. For each term, the two printed numbers should match.

In [12]:
# An empty list to hold one table per search term
tables = []             

# Repeat the search once for each of the related terms, changing only the search term
for term in related_terms:                         
    params["conditions[term]"] = '"' + term + '"'

    response = requests.get(api_url, params=params, timeout=60)
    response.raise_for_status()
    data = response.json()

    print("Term:", term)
    print("Status:", response.status_code)
    print("Response:", data)

    table = pd.DataFrame(data["results"])  # turn this term's rules into a table
    table["term"] = term         # add a "term" column so each row stores the term that found it
    tables.append(table)         # add the table to the list with all the other term tables

    print(term, ":", len(table), "of", data["count"], "rules downloaded") # print how many rules were downloaded and how many exist; the two should match

all_found = pd.concat(tables, ignore_index=True)  # stack all the tables into one, with new row numbers 0, 1, 2, ...
print("Total, including duplicates:", len(all_found))

Term: artificial intelligence
Status: 200
Response: {'description': 'Documents matching \'"artificial intelligence"\', published on or after 01/20/2017, and of type Rule', 'count': 190, 'total_pages': 1, 'results': [{'document_number': '2025-00636', 'title': 'Framework for Artificial Intelligence Diffusion', 'abstract': "With this interim final rule, the Commerce Department's Bureau of Industry and Security (BIS) revises the Export Administration Regulations' (EAR) controls on advanced computing integrated circuits (ICs) and adds a new control on artificial intelligence (AI) model weights for certain advanced closed-weight dual-use AI models. In conjunction with the expansion of these controls, which BIS has determined are necessary to protect U.S. national security and foreign policy interests, BIS is adding new license exceptions and updating the Data Center Validated End User authorization to facilitate the export, reexport, and transfer (in-country) of advanced computing (ICs) to e

#### Step 4: Remove duplicates

Some rules were found by more than one search term, so they appear more than once. This step keeps one copy of each rule and adds a `matched_terms` column that lists every term that found it (for example, "artificial intelligence; machine learning").

In [13]:
# For each rule, list all the terms that found it, separated by "; "
terms_per_rule = all_found.groupby("document_number")["term"].apply("; ".join)

# Keep one copy of each rule, then remove the "term" column
df = all_found.drop_duplicates(subset="document_number").drop(columns="term")

# Add a new column with each rule's joined terms, matched by document number
df["matched_terms"] = df["document_number"].map(terms_per_rule)

print("Unique rules:", len(df))
df[["document_number", "title", "matched_terms"]].head()

Unique rules: 228


,document_number,title,matched_terms
0,2025-00636,Framework for Artificial Intelligence Diffusion,artificial intelligence; GPUs
1,2025-00637,Public Briefing on Framework for Artificial In...,artificial intelligence
2,2023-05321,Copyright Registration Guidance: Works Contain...,artificial intelligence
3,2026-07087,Final Priority and Definitions-Secretary's Sup...,artificial intelligence; machine learning
4,2026-22593,Medical Devices; Hematology and Pathology Devi...,artificial intelligence; machine learning


#### Step 5: Agency names

Each rule lists the agencies behind it. This step adds two columns:

* `agency`: the first agency listed
* `all_agencies`: every agency listed

In [ ]:
# A function that takes one rule's list of agencies and returns their names as one piece of text

def get_agency_names(agencies):
    """Return every agency name for one rule, joined with "; "."""
    names = []                        # an empty list for the names
    for agency in agencies:           # go through each agency in the list
        names.append(agency.get("name") or agency.get("raw_name") or "Unknown")   # use "raw_name" if "name" is missing or "Unknown" if both are missing
    return "; ".join(names)           # join the names with "; " between them

# Run the function on every rule and save the result in a new column
df["all_agencies"] = df["agencies"].apply(get_agency_names)

# The first agency = the text before the first "; "
df["agency"] = df["all_agencies"].str.split("; ").str[0]

df = df.drop(columns="agencies")  # remove the original column; the two new columns replace it

df[["agency", "all_agencies"]].head()

,agency,all_agencies
0,Commerce Department,Commerce Department; Industry and Security Bureau
1,Commerce Department,Commerce Department; Industry and Security Bureau
2,Library of Congress,"Library of Congress; Copyright Office, Library..."
3,Education Department,Education Department
4,Health and Human Services Department,Health and Human Services Department; Food and...


#### Step 6: Administration labels

Each rule gets a label for the president's administration in office on the day the rule was published.

In [ ]:
# A function that takes one date and returns the administration in office on that date
def label_administration(date):
    """Return the administration in office on a given date."""
    if date < "2021-01-20":
        return "Trump I"        # Jan 20, 2017 to Jan 19, 2021
    elif date < "2025-01-20":
        return "Biden"          # Jan 20, 2021 to Jan 19, 2025
    else:
        return "Trump II"       # Jan 20, 2025 onward

# Run the function on every date in the publication_date column and save the result in a new column
df["administration"] = df["publication_date"].apply(label_administration)

# Count the rules for each administration
df["administration"].value_counts()

administration
Biden       114
Trump II     81
Trump I      29
Name: count, dtype: int64

#### Step 7: Remove corrections

A correction fixes an earlier rule, so it repeats that rule's content. Rules with titles ending in "Correction" are removed.

In [ ]:
# Check each title: True if it ends with "Correction", False if not
is_correction = df["title"].str.endswith("Correction")

print("Corrections removed:", is_correction.sum())     # count the True values

# Keep only the rules that are not corrections
df = df[is_correction == False]

print("Rules after removing corrections:", len(df))

Corrections removed: 1
Rules after removing corrections: 223


#### Step 8: Save

The table is saved as a CSV file (a simple spreadsheet file) so the next notebook can open it. `index=False` leaves out the row numbers.

In [ ]:
# Check that every rule appears only once
print("Every rule appears once:", df["document_number"].is_unique)

# Save the rules to a CSV file
df.to_csv("fr_rules.csv", index=False)
print("Saved", len(df), "rules to fr_rules.csv")

Every rule appears once: True
Saved 223 rules to fr_rules.csv


#### Step 9: Quick look at the data

A few counts to check that the data looks reasonable, and to spot anything worth noting on the slides.

In [ ]:
# Earliest and latest rule
print("First rule:", df["publication_date"].min())
print("Last rule: ", df["publication_date"].max())

# Count how many rules each combination of search terms found
df["matched_terms"].value_counts()

First rule: 2017-11-17
Last rule:  2026-11-09


matched_terms
artificial intelligence                                          122
artificial intelligence; machine learning                         56
machine learning                                                  29
artificial intelligence; large language model                      5
automated decision                                                 4
artificial intelligence; machine learning; deep learning           2
artificial intelligence; deep learning                             2
artificial intelligence; machine learning; automated decision      1
artificial intelligence; automated decision                        1
machine learning; deep learning                                    1
Name: count, dtype: int64

In [ ]:
# The 10 agencies with the most rules
df["agency"].value_counts().head(10)

agency
Health and Human Services Department    65
Commerce Department                     28
Homeland Security Department            16
Education Department                    12
Environmental Protection Agency         12
Consumer Financial Protection Bureau    11
Transportation Department               10
Securities and Exchange Commission       9
Federal Communications Commission        8
Energy Department                        7
Name: count, dtype: int64

#### Documentation

* **Federal Register API:** [overview](https://www.federalregister.gov/reader-aids/developer-resources/rest-api) · [search settings](https://www.federalregister.gov/developers/documentation/api/v1)
* **requests:** [quickstart](https://requests.readthedocs.io/en/latest/user/quickstart/)
* **pandas:** [concat](https://pandas.pydata.org/docs/reference/api/pandas.concat.html) · [groupby](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html) · [drop_duplicates](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop_duplicates.html) · [apply](https://pandas.pydata.org/docs/reference/api/pandas.Series.apply.html) · [str methods](https://pandas.pydata.org/docs/user_guide/text.html) · [selecting rows](https://pandas.pydata.org/docs/user_guide/indexing.html#boolean-indexing) · [to_csv](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html)
* **Python:** [lists](https://docs.python.org/3/tutorial/introduction.html#lists) · [for loops and if statements](https://docs.python.org/3/tutorial/controlflow.html) · [dict.get](https://docs.python.org/3/library/stdtypes.html#dict.get) · [str.join](https://docs.python.org/3/library/stdtypes.html#str.join)